# B2B 영업 데이터 전처리

이 노트북은 원본 22개 입력 컬럼을 확인하고, 서비스에서 공통으로 사용할 13개 모델 입력을 만든다.

- 사용자·조직별 값인 `Product`, `Seller`, `Att_t_client`를 제외한다.
- 미팅 기반 예측에 사용하지 않을 `Comp_size`, `Partnership`, `RFI`, `Growth`, `Strat_deal`, `Up_sale`을 제외한다.
- 원본 448행의 중복을 조사하되 제거하지 않고 모두 모델 데이터로 사용한다.
- `NaN`, 빈 문자열, 문자열 `Unknown`을 각각 센다.
- 세 유형을 합친 값을 **실질 결측치**로 사용한다.
- 컬럼별 결측률, 행별 평균 결측 개수, 범주 분포, `Unknown` 여부에 따른 Won 비율을 확인한다.
- 결측값은 대체하거나 삭제하지 않고 `Unknown` 범주로 유지한다.
- 실서비스에서 한 행의 입력값 약 30%가 확인되지 않을 것으로 가정해, 기존 `Unknown`을 포함해 각 행의 13개 컬럼 중 4개를 `Unknown`으로 마스킹한다.
- 원본 448행 각각에 Unknown 컬럼 조합이 겹치지 않는 마스킹 10세트를 먼저 만든다.
- 이후 동일한 13개 입력 조합과 그 마스킹 변형이 양쪽에 섞이지 않도록 그룹 단위로 Train/Test 7:3을 고정 분리한다.
- 분리된 13개 범주형 입력을 `pd.get_dummies(..., drop_first=True)`로 변환한다.

## 컬럼 설명

원본 컬럼명은 데이터 추적을 위해 임의로 바꾸지 않는다. 사용자·조직별 값이거나 실제 미팅 입력에서 사용하지 않을 9개 컬럼은 모델 입력에서 제외한다.

| 순서 | 컬럼 | 의미 | 데이터에 저장된 값 | 모델 사용 | 제외 이유 |
|---:|---|---|---|---|---|
| 1 | `Product` | 제안 상품 코드 | Product A~L, Product N, Product P | 제외 | 사용자·회사마다 상품 체계가 달라 공통 모델이 상품 코드를 암기할 위험이 있음 |
| 2 | `Seller` | 영업 담당자 코드 | Seller 1~17, Seller 20 | 제외 | 사용자·조직마다 담당자 식별자가 달라 다른 조직에 일반화되지 않음 |
| 3 | `Authority` | 고객 측 의사결정권자의 권한 수준 | High, Mid, Low | 사용 | - |
| 4 | `Comp_size` | 고객사 규모 | Big, Mid, Small | 제외 | 현재 서비스의 미팅 기반 승패 판단에 고객사 규모를 사용하지 않기로 함 |
| 5 | `Competitors` | 경쟁사 존재 또는 검토 여부 | Yes, No, Unknown | 사용 | - |
| 6 | `Purch_dept` | 고객사 구매부서 참여 여부 | Yes, No, Unknown | 사용 | - |
| 7 | `Partnership` | 파트너와 협업해 판매하는 딜인지 여부 | Yes, No | 제외 | 파트너십 여부를 이번 서비스의 승패 판단 요소로 사용하지 않기로 함 |
| 8 | `Budgt_alloc` | 고객 예산 확보·배정 여부 | Yes, No, Unknown | 사용 | - |
| 9 | `Forml_tend` | 공식 입찰 절차 여부 | Yes, No | 사용 | - |
| 10 | `RFI` | 정보요청서(Request for Information) 진행 여부 | Yes, No | 제외 | 정보요청서 진행 여부를 이번 서비스의 승패 판단 요소로 사용하지 않기로 함 |
| 11 | `RFP` | 제안요청서(Request for Proposal) 진행 여부 | Yes, No | 사용 | - |
| 12 | `Growth` | 고객사의 성장 상태 | Growth, Stable, Slow down, Unknown | 제외 | 고객사의 성장 상태를 이번 서비스의 승패 판단 요소로 사용하지 않기로 함 |
| 13 | `Posit_statm` | 고객의 명시적인 긍정 구매 표현 여부 | Yes, No, Neutral | 사용 | - |
| 14 | `Source` | 영업기회가 유입된 경로 | Direct mail, Event, Joint past, Media, Online form, Other, Referral, Unknown | 사용 | - |
| 15 | `Client` | 고객과의 거래 관계 | Current, New, Past | 사용 | - |
| 16 | `Scope` | 계약·수행 범위의 명확성 | Clear, Few questions, Low | 사용 | - |
| 17 | `Strat_deal` | 회사 관점에서 딜의 전략적 중요도 | Very important, Average important, Unimportant | 제외 | 미팅 내용에서 안정적으로 추출하기 어려운 내부 판단값임 |
| 18 | `Cross_sale` | 기존 고객에게 다른 상품을 함께 판매하는 교차판매 여부 | Yes, No | 사용 | - |
| 19 | `Up_sale` | 더 높은 등급·규모의 상품을 제안하는 상향판매 여부 | Yes, No | 제외 | 서비스에 상품 간 등급·서열 데이터가 없어 일관되게 판정할 수 없음 |
| 20 | `Deal_type` | 거래의 유형 | Consulting, Maintenance, Project, Solution | 사용 | - |
| 21 | `Needs_def` | 고객 요구사항의 정의 수준 | Yes, Poor, Info gathering, No | 사용 | - |
| 22 | `Att_t_client` | 영업 조직이 분류한 고객 관리 유형 | Bad client, First deal, Normal, Strategic account | 제외 | 조직마다 정의하는 주관적 관리 유형이라 다른 사용자·조직에 일반화되지 않음 |
| 정답 | `Status` | 딜의 최종 성사 여부 | Won, Lost | 정답 | - |

`Unknown`은 `No`와 다르다. `No`는 해당 사실이 없다고 확인된 값이고, `Unknown`은 해당 사실을 측정하거나 확인하지 못한 값이다. 원본에서 `Unknown`이 없던 컬럼도 실서비스에서는 값이 추출되지 않을 수 있으므로 13개 모델 입력 모두 `Unknown`을 허용한다.

In [1]:
import hashlib
import os
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import StratifiedGroupKFold

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)

# 환경변수를 지정하면 다른 위치의 동일 데이터셋을 사용할 수 있다.
DATA_PATH = (
    Path(
        os.environ.get(
            "SALESLUV_B2B_DATA_PATH",
            "/private/tmp/Salvirt_B2B_ML_dataset_HF.csv",
        )
    )
    .expanduser()
    .resolve()
)
SOURCE_SHA256 = "8dee635b95bdcb00896b654efe62fc20177090081c81ef5224e8641ba31c3061"

SOURCE_FEATURE_NAMES = (
    "Product",
    "Seller",
    "Authority",
    "Comp_size",
    "Competitors",
    "Purch_dept",
    "Partnership",
    "Budgt_alloc",
    "Forml_tend",
    "RFI",
    "RFP",
    "Growth",
    "Posit_statm",
    "Source",
    "Client",
    "Scope",
    "Strat_deal",
    "Cross_sale",
    "Up_sale",
    "Deal_type",
    "Needs_def",
    "Att_t_client",
)
EXCLUDED_FEATURE_NAMES = (
    "Product",
    "Seller",
    "Comp_size",
    "Partnership",
    "RFI",
    "Growth",
    "Strat_deal",
    "Up_sale",
    "Att_t_client",
)
MODEL_FEATURE_NAMES = tuple(
    column for column in SOURCE_FEATURE_NAMES if column not in EXCLUDED_FEATURE_NAMES
)
CATEGORY_VALUES = {
    "Authority": ("High", "Low", "Mid", "Unknown"),
    "Competitors": ("No", "Unknown", "Yes"),
    "Purch_dept": ("No", "Unknown", "Yes"),
    "Budgt_alloc": ("No", "Unknown", "Yes"),
    "Forml_tend": ("No", "Unknown", "Yes"),
    "RFP": ("No", "Unknown", "Yes"),
    "Posit_statm": ("Neutral", "No", "Unknown", "Yes"),
    "Source": (
        "Direct mail",
        "Event",
        "Joint past",
        "Media",
        "Online form",
        "Other",
        "Referral",
        "Unknown",
    ),
    "Client": ("Current", "New", "Past", "Unknown"),
    "Scope": ("Clear", "Few questions", "Low", "Unknown"),
    "Cross_sale": ("No", "Unknown", "Yes"),
    "Deal_type": ("Consulting", "Maintenance", "Project", "Solution", "Unknown"),
    "Needs_def": ("Info gathering", "No", "Poor", "Unknown", "Yes"),
}
MASKING_RATE = 0.30
MASKING_SET_COUNT = 10
MASKING_RANDOM_STATE = 1
UNKNOWN_COLUMNS_PER_ROW = round(len(MODEL_FEATURE_NAMES) * MASKING_RATE)
TARGET_NAME = "Status"
ALL_COLUMNS = (*SOURCE_FEATURE_NAMES, TARGET_NAME)
MODEL_DEDUPLICATION_COLUMNS = (*MODEL_FEATURE_NAMES, TARGET_NAME)

assert len(SOURCE_FEATURE_NAMES) == 22
assert len(MODEL_FEATURE_NAMES) == 13
assert tuple(CATEGORY_VALUES) == MODEL_FEATURE_NAMES
assert UNKNOWN_COLUMNS_PER_ROW == 4
assert MASKING_SET_COUNT == 10

In [2]:
# 구분자를 지정하지 않으면 세미콜론 CSV가 한 컬럼으로 읽히므로 명시적으로 지정한다.
assert DATA_PATH.exists(), f"데이터 파일이 없습니다: {DATA_PATH}"
source_sha256 = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()
assert source_sha256 == SOURCE_SHA256, (
    "예상한 Salvirt 데이터셋과 파일 해시가 다릅니다. "
    f"expected={SOURCE_SHA256}, actual={source_sha256}"
)

data_raw = pd.read_csv(DATA_PATH, sep=";", dtype="string")
data_raw.columns = data_raw.columns.str.strip()
assert tuple(data_raw.columns) == ALL_COLUMNS, (
    f"컬럼 이름이나 순서가 예상과 다릅니다. actual={tuple(data_raw.columns)}"
)

print(f"데이터 경로: {DATA_PATH}")
print(f"SHA-256: {source_sha256}")
print(f"원본 크기: {data_raw.shape[0]}행 × {data_raw.shape[1]}컬럼")

데이터 경로: 로컬 원본 파일(PR 제외)
SHA-256: 8dee635b95bdcb00896b654efe62fc20177090081c81ef5224e8641ba31c3061
원본 크기: 448행 × 23컬럼


### 해석

- 파일 해시가 고정된 원본과 일치하므로 이전 조사와 같은 데이터셋을 보고 있다.
- 데이터는 448행, 23컬럼이며 22개 입력 컬럼과 정답 `Status` 1개로 구성된다.
- 원본 구조는 유지하되 제외하기로 정한 9개 컬럼을 빼면 실제 모델 후보는 13개다.
- 해시나 컬럼 순서가 다르면 이후 수치의 비교 기준이 달라지므로 이 단계에서 실행을 중단하도록 했다.

## 1. 원본 데이터 확인

먼저 컬럼이 올바르게 분리되었는지와 실제 저장값의 형태를 확인한다.

In [3]:
display(data_raw.head(10))

### 해석

- 모든 입력값은 숫자가 아니라 범주형 문자열이다. 따라서 평균값 대체나 수치형 중앙값 대체를 바로 적용할 수 없다.
- 일부 셀의 `Unknown`은 CSV의 빈칸이나 `NaN`이 아니라 실제 문자열로 저장되어 있다.
- 상위 10행은 값의 형태를 확인하는 용도이며, 결측 비율 판단에는 전체 448행을 사용한다.

In [4]:
# 앞뒤 공백은 범주가 아니라 입력 오류이므로 결측 조사 전에 문자열만 정리한다.
# 이 셀은 값 대체나 행 삭제를 하지 않는다.
data_clean = data_raw.copy()
for column in data_clean.columns:
    data_clean[column] = data_clean[column].str.strip()

full_duplicate_mask = data_clean.duplicated(keep="first")
model_duplicate_mask = data_clean.duplicated(subset=list(MODEL_DEDUPLICATION_COLUMNS), keep="first")
data_model = data_clean.reset_index(drop=True)
conflicting_input_groups = (
    data_model.groupby(list(MODEL_FEATURE_NAMES), dropna=False)[TARGET_NAME].nunique().gt(1).sum()
)

dataset_overview = pd.DataFrame(
    {
        "항목": [
            "원본 행 수",
            "전체 컬럼 수",
            "원본 입력 컬럼 수",
            "제외 입력 컬럼 수",
            "모델 입력 컬럼 수",
            "23개 전체 컬럼 완전중복 행 수",
            "13개 입력+Status 기준 중복 행 수",
            "13개 입력+Status 기준 고유 행 수",
            "중복을 유지한 모델 사용 행 수",
            "동일한 13개 입력에 Won·Lost가 모두 있는 조합 수",
        ],
        "값": [
            len(data_clean),
            data_clean.shape[1],
            len(SOURCE_FEATURE_NAMES),
            len(EXCLUDED_FEATURE_NAMES),
            len(MODEL_FEATURE_NAMES),
            int(full_duplicate_mask.sum()),
            int(model_duplicate_mask.sum()),
            int((~model_duplicate_mask).sum()),
            len(data_model),
            int(conflicting_input_groups),
        ],
    }
)
display(dataset_overview)
display(
    data_model[TARGET_NAME].value_counts(dropna=False).rename_axis(TARGET_NAME).to_frame("행 수")
)

,항목,값
0,원본 행 수,448
1,전체 컬럼 수,23
2,원본 입력 컬럼 수,22
3,제외 입력 컬럼 수,9
4,모델 입력 컬럼 수,13
5,23개 전체 컬럼 완전중복 행 수,83
6,13개 입력+Status 기준 중복 행 수,226
7,13개 입력+Status 기준 고유 행 수,222
8,중복을 유지한 모델 사용 행 수,448
9,동일한 13개 입력에 Won·Lost가 모두 있는 조합 수,24


,행 수
Status,
Won,227
Lost,221


### 해석

- 23개 전체 컬럼 기준 완전중복은 83행이지만, 제외 컬럼을 빼고 13개 입력과 `Status`로 보면 중복은 226행이다.
- 13개 입력과 `Status`가 같은 고유 행은 222개지만, 반복 행도 서로 다른 영업 사례로 보고 448행을 모두 유지한다.
- 전체 정답은 `Won` 227건(50.67%), `Lost` 221건(49.33%)으로 거의 균형이다.
- 동일한 13개 입력인데 정답이 Won과 Lost로 갈리는 조합은 24개다. 현재 입력만으로 구분되지 않는 사례도 그대로 유지한다.
- 행별 무작위 마스킹은 반복 행의 관측 형태를 다르게 만들지만 원본 사례의 독립성을 새로 만들어내는 것은 아니므로 평가 결과 해석 시 중복 비중을 함께 고려한다.

## 2. 결측치 정의와 컬럼별 조사

범주형 데이터에는 평균값을 바로 넣을 수 없다. 먼저 아래 세 유형을 구분해서 센다.

1. `NaN`: CSV 파싱 결과 실제 null
2. 빈 문자열: 값이 없거나 공백만 있던 셀
3. `Unknown`: 데이터 제공자가 측정하지 못했다고 명시한 범주

이 셋의 합집합을 실질 결측치로 정의한다.

In [5]:
def missing_masks(data: pd.DataFrame) -> dict[str, pd.DataFrame]:
    """NaN, 빈 문자열, Unknown과 세 유형의 합집합을 반환한다."""
    null_mask = data.isna()
    blank_mask = data.notna() & data.eq("")
    unknown_mask = data.apply(lambda column: column.str.casefold().eq("unknown").fillna(False))
    effective_mask = null_mask | blank_mask | unknown_mask
    return {
        "null": null_mask,
        "blank": blank_mask,
        "unknown": unknown_mask,
        "effective": effective_mask,
    }


def missing_profile(data: pd.DataFrame) -> pd.DataFrame:
    """각 컬럼의 범주 수와 결측 유형별 개수·비율을 계산한다."""
    masks = missing_masks(data)
    observed_category_counts = {
        column: data.loc[~masks["effective"][column], column].nunique(dropna=True)
        for column in data.columns
    }
    profile = pd.DataFrame(
        {
            "관측_범주_수": pd.Series(observed_category_counts),
            "NaN_수": masks["null"].sum(),
            "빈문자열_수": masks["blank"].sum(),
            "Unknown_수": masks["unknown"].sum(),
            "실질_결측_수": masks["effective"].sum(),
            "실질_결측률(%)": masks["effective"].mean().mul(100),
        }
    )
    profile.index.name = "컬럼"
    return profile


profile_model = missing_profile(data_model)

In [6]:
# 중복을 포함해 실제 학습에 사용할 448행의 자연 결측 상태를 확인한다.
natural_missingness = profile_model.loc[
    list(MODEL_FEATURE_NAMES),
    ["실질_결측_수", "실질_결측률(%)"],
]
display(natural_missingness.round(3))

,실질_결측_수,실질_결측률(%)
컬럼,,
Authority,0,0.0
Competitors,20,4.464
Purch_dept,32,7.143
Budgt_alloc,36,8.036
Forml_tend,0,0.0
RFP,0,0.0
Posit_statm,0,0.0
Source,10,2.232
Client,0,0.0


### 해석

- 실제 `NaN`과 빈 문자열은 없고 실질 결측치는 모두 문자열 `Unknown`이다.
- 13개 모델 입력 중 `Competitors`, `Purch_dept`, `Budgt_alloc`, `Source`에서만 자연 `Unknown`이 발견된다.
- 중복을 제거하지 않기로 했으므로 이후 결측 분석과 마스킹은 모두 448행을 기준으로 한다.

In [7]:
# 이후 처리 결정을 위한 기준표는 실제 모델에 사용할 448행으로 정렬한다.
feature_missingness = profile_model.loc[list(MODEL_FEATURE_NAMES)].sort_values(
    ["실질_결측률(%)", "실질_결측_수"], ascending=False
)
display(feature_missingness.round(3))

,관측_범주_수,NaN_수,빈문자열_수,Unknown_수,실질_결측_수,실질_결측률(%)
컬럼,,,,,,
Budgt_alloc,2,0,0,36,36,8.036
Purch_dept,2,0,0,32,32,7.143
Competitors,2,0,0,20,20,4.464
Source,7,0,0,10,10,2.232
Authority,3,0,0,0,0,0.0
Forml_tend,2,0,0,0,0,0.0
RFP,2,0,0,0,0,0.0
Posit_statm,3,0,0,0,0,0.0
Client,3,0,0,0,0,0.0


### 해석

- `Budgt_alloc` 36건(8.04%), `Purch_dept` 32건(7.14%), `Competitors` 20건(4.46%), `Source` 10건(2.23%) 순이다.
- 나머지 9개 모델 입력 컬럼에는 실질 결측치가 없다.
- 자연 결측률은 최대 8.04%다. 실서비스 결측 가정은 뒤에서 행별 4개 `Unknown` 마스킹으로 별도 반영한다.

## 3. 전체 평균과 행별 결측 개수

컬럼별 비율뿐 아니라 한 행에 `Unknown`이 평균 몇 개 들어 있는지도 확인한다. 여러 컬럼이 같은 행에서 함께 비는 경우 단일 컬럼 대체보다 행 전체의 정보 부족 문제로 봐야 한다.

In [8]:
def dataset_missing_summary(name: str, data: pd.DataFrame) -> dict[str, float | int | str]:
    """13개 모델 입력 컬럼 전체의 결측 상태를 한 행으로 요약한다."""
    masks = missing_masks(data)
    feature_mask = masks["effective"].loc[:, list(MODEL_FEATURE_NAMES)]
    row_missing_counts = feature_mask.sum(axis=1)
    return {
        "데이터": name,
        "행_수": len(data),
        "전체_입력셀_수": data.shape[0] * len(MODEL_FEATURE_NAMES),
        "실질_결측셀_수": int(feature_mask.to_numpy().sum()),
        "전체_입력셀_결측률(%)": float(feature_mask.to_numpy().mean() * 100),
        "결측_컬럼_수": int(feature_mask.any(axis=0).sum()),
        "결측_포함_행_수": int(row_missing_counts.gt(0).sum()),
        "결측_포함_행_비율(%)": float(row_missing_counts.gt(0).mean() * 100),
        "행당_평균_결측_개수": float(row_missing_counts.mean()),
        "행당_중앙_결측_개수": float(row_missing_counts.median()),
        "행당_최대_결측_개수": int(row_missing_counts.max()),
    }


missing_summary = pd.DataFrame([dataset_missing_summary("모델 사용 전체", data_model)])
display(missing_summary.round(3))

,데이터,행_수,전체_입력셀_수,실질_결측셀_수,전체_입력셀_결측률(%),결측_컬럼_수,결측_포함_행_수,결측_포함_행_비율(%),행당_평균_결측_개수,행당_중앙_결측_개수,행당_최대_결측_개수
0,모델 사용 전체,448,5824,98,1.683,4,72,16.071,0.219,0.0,3


### 해석

- 전체 모델 입력 5,824셀 중 자연 `Unknown`은 98셀로 셀 기준 결측률은 1.683%다.
- 행 기준으로는 448행 중 72행(16.07%)이 `Unknown`을 하나 이상 포함한다.
- 행당 평균은 0.219개, 중앙값은 0개, 최대는 3개다. 이 자연 결측은 유지하고 각 행이 총 4개 `Unknown`을 갖도록 뒤에서 추가 마스킹한다.

In [9]:
model_masks = missing_masks(data_model)
row_missing_counts = model_masks["effective"][list(MODEL_FEATURE_NAMES)].sum(axis=1)
row_missing_distribution = (
    row_missing_counts.value_counts().sort_index().rename_axis("행당_결측_개수").to_frame("행_수")
)
row_missing_distribution["비율(%)"] = row_missing_distribution["행_수"] / len(data_model) * 100
display(row_missing_distribution.round(3))

,행_수,비율(%)
행당_결측_개수,,
0,376,83.929
1,49,10.938
2,20,4.464
3,3,0.67


### 해석

- 자연 `Unknown`이 없는 행은 376건(83.93%)이다.
- 1개를 포함한 행은 49건, 2개는 20건, 3개는 3건이다.
- 기존 `Unknown`이 가장 많은 행도 3개이므로 모든 행을 기존 값의 손실 없이 총 4개 `Unknown`으로 맞출 수 있다.

## 4. `Unknown` 컬럼의 범주와 정답 분포

최빈값으로 채우기 전에 `Unknown`이 있는 행과 값이 관측된 행의 Won 비율을 비교한다. 두 집단의 차이가 크면 `Unknown` 자체가 정보일 수 있으므로 무조건 대체하면 신호가 사라질 수 있다. 이 표는 인과관계를 증명하지 않고 차이만 보여준다.

In [10]:
unknown_columns = [column for column in MODEL_FEATURE_NAMES if model_masks["unknown"][column].any()]
print(f"Unknown이 있는 컬럼 ({len(unknown_columns)}개): {unknown_columns}")

category_distribution_rows = []
for column in unknown_columns:
    counts = data_model[column].value_counts(dropna=False)
    for value, count in counts.items():
        category_distribution_rows.append(
            {
                "컬럼": column,
                "값": value,
                "행_수": int(count),
                "비율(%)": float(count / len(data_model) * 100),
            }
        )

category_distribution = pd.DataFrame(category_distribution_rows)
display(category_distribution.round(3))

Unknown이 있는 컬럼 (4개): ['Competitors', 'Purch_dept', 'Budgt_alloc', 'Source']


,컬럼,값,행_수,비율(%)
0,Competitors,No,285,63.616
1,Competitors,Yes,143,31.920
2,Competitors,Unknown,20,4.464
3,Purch_dept,No,302,67.411
4,Purch_dept,Yes,114,25.446
5,Purch_dept,Unknown,32,7.143
6,Budgt_alloc,Yes,219,48.884
7,Budgt_alloc,No,193,43.080
8,Budgt_alloc,Unknown,36,8.036
9,Source,Joint past,339,75.670


### 해석

- `Unknown`은 4개 컬럼 모두에서 가장 적은 범주다.
- `Source`는 `Joint past`가 339/448건으로 특정 범주 비중이 매우 높다.
- 이런 컬럼을 최빈값으로 채우면 대부분의 `Unknown`이 동일 범주로 바뀌므로 실제 관측값과 추정값을 구분할 수 없게 된다.

In [11]:
target = data_model[TARGET_NAME].map({"Lost": 0, "Won": 1})
assert target.notna().all(), "Status에는 Lost와 Won만 있어야 합니다."

unknown_target_rows = []
for column in unknown_columns:
    is_unknown = model_masks["unknown"][column]
    unknown_won_rate = target.loc[is_unknown].mean()
    observed_won_rate = target.loc[~is_unknown].mean()
    unknown_target_rows.append(
        {
            "컬럼": column,
            "Unknown_행_수": int(is_unknown.sum()),
            "Unknown_Won_비율(%)": float(unknown_won_rate * 100),
            "관측_Won_비율(%)": float(observed_won_rate * 100),
            "차이(%p)": float((unknown_won_rate - observed_won_rate) * 100),
        }
    )

unknown_target_comparison = (
    pd.DataFrame(unknown_target_rows)
    .sort_values("Unknown_행_수", ascending=False)
    .reset_index(drop=True)
)
display(unknown_target_comparison.round(3))

# Unknown이 하나라도 있는 행을 모두 삭제할 경우 사라지는 정답 분포를 확인한다.
has_any_unknown = model_masks["unknown"][list(MODEL_FEATURE_NAMES)].any(axis=1)
row_level_target_comparison = pd.DataFrame(
    [
        {
            "구분": "Unknown 없음",
            "행_수": int((~has_any_unknown).sum()),
            "Lost_수": int(data_model.loc[~has_any_unknown, TARGET_NAME].eq("Lost").sum()),
            "Won_수": int(data_model.loc[~has_any_unknown, TARGET_NAME].eq("Won").sum()),
            "Won_비율(%)": float(target.loc[~has_any_unknown].mean() * 100),
        },
        {
            "구분": "Unknown 1개 이상",
            "행_수": int(has_any_unknown.sum()),
            "Lost_수": int(data_model.loc[has_any_unknown, TARGET_NAME].eq("Lost").sum()),
            "Won_수": int(data_model.loc[has_any_unknown, TARGET_NAME].eq("Won").sum()),
            "Won_비율(%)": float(target.loc[has_any_unknown].mean() * 100),
        },
    ]
)
display(row_level_target_comparison.round(3))

,컬럼,Unknown_행_수,Unknown_Won_비율(%),관측_Won_비율(%),차이(%p)
0,Budgt_alloc,36,13.889,53.883,-39.995
1,Purch_dept,32,15.625,53.365,-37.740
2,Competitors,20,25.000,51.869,-26.869
3,Source,10,20.000,51.370,-31.370


,구분,행_수,Lost_수,Won_수,Won_비율(%)
0,Unknown 없음,376,163,213,56.649
1,Unknown 1개 이상,72,58,14,19.444


### 해석

- 4개 컬럼 모두 자연 `Unknown` 행의 Won 비율이 관측 행보다 26.87~39.99%p 낮다.
- 자연 `Unknown`이 하나 이상 있는 72행은 `Lost` 58건, `Won` 14건으로 Won 비율이 19.44%다. `Unknown`이 없는 376행의 Won 비율은 56.65%다.
- 따라서 현재 데이터에서 `Unknown`은 무작위 결측처럼 보이지 않는다. 행을 삭제하면 Lost를 주로 제거하고, 최빈값으로 대체하면 실패 가능성과 연결된 신호를 지울 수 있다.
- 다만 `Unknown` 표본이 적으므로 이 차이는 연관성으로만 해석하며 원인으로 단정하지 않는다.

In [12]:
# 어떤 Unknown 컬럼들이 같은 행에서 함께 나타나는지 확인한다.
unknown_matrix = model_masks["unknown"][unknown_columns].astype(int)
unknown_cooccurrence = unknown_matrix.T.dot(unknown_matrix)
display(unknown_cooccurrence)

unknown_combinations = (
    unknown_matrix.loc[unknown_matrix.sum(axis=1).gt(0)]
    .apply(
        lambda row: ", ".join(row.index[row.eq(1)]),
        axis=1,
    )
    .value_counts()
    .rename_axis("동시에_Unknown인_컬럼")
    .to_frame("행_수")
)
display(unknown_combinations)

,Competitors,Purch_dept,Budgt_alloc,Source
Competitors,20,3,6,4
Purch_dept,3,32,14,0
Budgt_alloc,6,14,36,2
Source,4,0,2,10


,행_수
동시에_Unknown인_컬럼,
Budgt_alloc,17
Purch_dept,16
"Purch_dept, Budgt_alloc",13
Competitors,10
Source,6
"Competitors, Budgt_alloc",3
"Competitors, Source",2
"Competitors, Purch_dept",2
"Competitors, Budgt_alloc, Source",2


### 해석

- 단독 발생은 `Budgt_alloc` 17행, `Purch_dept` 16행, `Competitors` 10행, `Source` 6행이다.
- `Purch_dept`와 `Budgt_alloc`이 함께 `Unknown`인 행은 14행이며, 이 중 두 컬럼만 비어 있는 행은 13행이다.
- 여러 컬럼이 항상 한꺼번에 누락되는 구조는 아니지만 구매부서·예산 관련 값은 함께 확인되지 않는 경향이 있어 처리 시 둘의 관계를 보존해야 한다.

## 5. 처리 방법 결정표

13개 입력 모두 실서비스에서 값이 추출되지 않을 수 있으므로 `Unknown`을 허용한다. 원본에 이미 있는 `Unknown`은 정답과 연관된 정보를 가지므로 `No`나 최빈값으로 바꾸지 않는다.

In [13]:
missingness_review = feature_missingness.reset_index().merge(
    unknown_target_comparison,
    on="컬럼",
    how="left",
)
has_missing = missingness_review["실질_결측_수"].gt(0)
missingness_review["처리_결정"] = "Unknown 범주 허용"
missingness_review["결정_근거"] = "실서비스 미팅에서 값이 추출되지 않을 가능성을 반영"
missingness_review.loc[has_missing, "처리_결정"] = "기존 Unknown 유지"
missingness_review.loc[has_missing, "결정_근거"] = (
    "Unknown 행과 관측 행의 Won 비율이 달라 별도 범주 신호를 보존"
)
display(missingness_review.round(3))

,컬럼,관측_범주_수,NaN_수,빈문자열_수,Unknown_수,실질_결측_수,실질_결측률(%),Unknown_행_수,Unknown_Won_비율(%),관측_Won_비율(%),차이(%p),처리_결정,결정_근거
0,Budgt_alloc,2,0,0,36,36,8.036,36.0,13.889,53.883,-39.995,기존 Unknown 유지,Unknown 행과 관측 행의 Won 비율이 달라 별도 범주 신호를 보존
1,Purch_dept,2,0,0,32,32,7.143,32.0,15.625,53.365,-37.740,기존 Unknown 유지,Unknown 행과 관측 행의 Won 비율이 달라 별도 범주 신호를 보존
2,Competitors,2,0,0,20,20,4.464,20.0,25.000,51.869,-26.869,기존 Unknown 유지,Unknown 행과 관측 행의 Won 비율이 달라 별도 범주 신호를 보존
3,Source,7,0,0,10,10,2.232,10.0,20.000,51.370,-31.370,기존 Unknown 유지,Unknown 행과 관측 행의 Won 비율이 달라 별도 범주 신호를 보존
4,Authority,3,0,0,0,0,0.0,NaN,NaN,NaN,NaN,Unknown 범주 허용,실서비스 미팅에서 값이 추출되지 않을 가능성을 반영
5,Forml_tend,2,0,0,0,0,0.0,NaN,NaN,NaN,NaN,Unknown 범주 허용,실서비스 미팅에서 값이 추출되지 않을 가능성을 반영
6,RFP,2,0,0,0,0,0.0,NaN,NaN,NaN,NaN,Unknown 범주 허용,실서비스 미팅에서 값이 추출되지 않을 가능성을 반영
7,Posit_statm,3,0,0,0,0,0.0,NaN,NaN,NaN,NaN,Unknown 범주 허용,실서비스 미팅에서 값이 추출되지 않을 가능성을 반영
8,Client,3,0,0,0,0,0.0,NaN,NaN,NaN,NaN,Unknown 범주 허용,실서비스 미팅에서 값이 추출되지 않을 가능성을 반영
9,Scope,3,0,0,0,0,0.0,NaN,NaN,NaN,NaN,Unknown 범주 허용,실서비스 미팅에서 값이 추출되지 않을 가능성을 반영


### 해석

- `Competitors`, `Purch_dept`, `Budgt_alloc`, `Source`에 원래 있던 `Unknown`은 별도 범주로 유지한다.
- 원본에 `Unknown`이 없던 나머지 9개 컬럼도 실서비스의 값 미추출 상황을 반영할 수 있도록 `Unknown` 범주를 허용한다.
- 행 삭제나 최빈값 대체를 하지 않으므로 기존 `Unknown`과 Lost의 연관 신호 및 448개 모델 대상 행을 보존한다.

## 6. 범주 스키마와 행별 30% `Unknown` 마스킹

구조화 Agent는 아래 13개 컬럼과 허용 범주만 출력한다. 실서비스에서 한 행의 입력값 약 30%가 확인되지 않을 것으로 가정하므로, 기존 `Unknown`을 포함해 각 행마다 4개 컬럼을 `Unknown`으로 만든다. 각 행에서 가능한 컬럼 조합을 구한 뒤 비복원 추출해 서로 겹치지 않는 마스킹 10세트를 만든다.

In [14]:
# 고정된 범주 외 값이 있으면 오탈자나 구조화 오류일 수 있으므로 즉시 중단한다.
invalid_category_values = {}
for column, allowed_values in CATEGORY_VALUES.items():
    actual_values = set(data_model[column].dropna().unique())
    invalid_values = sorted(actual_values - set(allowed_values))
    if invalid_values:
        invalid_category_values[column] = invalid_values
assert not invalid_category_values, f"허용되지 않은 범주가 있습니다: {invalid_category_values}"

category_schema = pd.DataFrame(
    [
        {
            "컬럼": column,
            "허용_범주": ", ".join(allowed_values),
            "범주_수": len(allowed_values),
            "기준_범주(drop_first)": allowed_values[0],
        }
        for column, allowed_values in CATEGORY_VALUES.items()
    ]
)
display(category_schema)


def unique_mask_counts_per_row(masked_sets: dict[str, pd.DataFrame]) -> pd.Series:
    """각 원본 행이 세트 전체에서 가진 서로 다른 Unknown 조합 수를 센다."""
    unknown_masks = [masked.eq("Unknown") for masked in masked_sets.values()]
    first_index = next(iter(masked_sets.values())).index
    return pd.Series(
        {
            row_index: len(
                {tuple(mask.loc[row_index].to_numpy(dtype=bool)) for mask in unknown_masks}
            )
            for row_index in first_index
        },
        name="고유_마스킹_조합_수",
    )


def make_unique_masked_sets(
    data: pd.DataFrame,
    set_prefix: str,
    random_state: int,
) -> dict[str, pd.DataFrame]:
    """각 행에서 겹치지 않는 Unknown 조합 10개를 만들어 세트별로 반환한다."""
    set_names = tuple(
        f"{set_prefix}_mask_{set_number:02d}" for set_number in range(1, MASKING_SET_COUNT + 1)
    )
    masked_sets = {set_name: data.copy() for set_name in set_names}
    existing_unknown_mask = data.eq("Unknown")
    existing_row_unknown_counts = existing_unknown_mask.sum(axis=1)
    assert existing_row_unknown_counts.le(UNKNOWN_COLUMNS_PER_ROW).all()

    rng = np.random.default_rng(random_state)
    for row_index in data.index:
        columns_to_mask = UNKNOWN_COLUMNS_PER_ROW - int(existing_row_unknown_counts.loc[row_index])
        observed_columns = tuple(
            data.columns[~existing_unknown_mask.loc[row_index].to_numpy(dtype=bool)]
        )
        possible_masks = tuple(combinations(observed_columns, columns_to_mask))
        assert len(possible_masks) >= MASKING_SET_COUNT
        selected_mask_indices = rng.choice(
            len(possible_masks),
            size=MASKING_SET_COUNT,
            replace=False,
        )

        for set_name, mask_index in zip(set_names, selected_mask_indices, strict=True):
            columns_to_replace = list(possible_masks[int(mask_index)])
            masked_sets[set_name].loc[row_index, columns_to_replace] = "Unknown"

    for masked in masked_sets.values():
        for column, allowed_values in CATEGORY_VALUES.items():
            masked[column] = pd.Categorical(masked[column], categories=allowed_values)
        final_unknown_mask = masked.eq("Unknown")
        assert (existing_unknown_mask <= final_unknown_mask).all().all()
        assert final_unknown_mask.sum(axis=1).eq(UNKNOWN_COLUMNS_PER_ROW).all()

    assert unique_mask_counts_per_row(masked_sets).eq(MASKING_SET_COUNT).all()
    return masked_sets


def encode_categories(data: pd.DataFrame) -> pd.DataFrame:
    """고정 범주 순서로 13개 입력을 동일한 원핫 컬럼으로 변환한다."""
    return pd.get_dummies(
        data,
        columns=list(MODEL_FEATURE_NAMES),
        drop_first=True,
    )

,컬럼,허용_범주,범주_수,기준_범주(drop_first)
0,Authority,"High, Low, Mid, Unknown",4,High
1,Competitors,"No, Unknown, Yes",3,No
2,Purch_dept,"No, Unknown, Yes",3,No
3,Budgt_alloc,"No, Unknown, Yes",3,No
4,Forml_tend,"No, Unknown, Yes",3,No
5,RFP,"No, Unknown, Yes",3,No
6,Posit_statm,"Neutral, No, Unknown, Yes",4,Neutral
7,Source,"Direct mail, Event, Joint past, Media, Online ...",8,Direct mail
8,Client,"Current, New, Past, Unknown",4,Current
9,Scope,"Clear, Few questions, Low, Unknown",4,Clear


### 해석

- 13개 입력 컬럼의 허용 범주를 고정했고, 모든 컬럼에서 값 미확인을 나타내는 `Unknown`을 허용한다. 표의 첫 범주는 `drop_first=True`에서 제외되는 기준 범주다.
- 13개 컬럼의 정확한 30%는 3.9개이므로 가장 가까운 정수인 4개를 사용한다.
- `make_unique_masked_sets`는 기존 `Unknown`을 보존하고 각 행에서 부족한 컬럼의 조합을 만든 뒤 10개를 중복 없이 선택한다.
- 자연 `Unknown`이 3개인 행도 남은 10개 컬럼 중 1개를 고르는 조합이 10개이므로 모든 행에서 서로 다른 10세트를 만들 수 있다. 정답은 마스킹 위치 결정에 사용하지 않는다.
- `encode_categories`는 고정 범주 순서와 `drop_first=True`를 사용하므로 모든 Train/Test 마스킹 세트가 동일한 39개 컬럼으로 변환된다.

## 7. 전체 448행 마스킹 후 그룹 단위 Train/Test 분리

정답은 `Lost=0`, `Won=1`로 변환한다. 먼저 원본 448행마다 서로 겹치지 않는 마스킹 조합 10개를 만든다. 이후 중복 행은 삭제하지 않되 동일한 13개 입력 조합과 그 10개 변형을 한 그룹으로 유지하며 Train/Test를 나눈다.

In [15]:
# 정답만 0과 1로 변환하고 입력 범주는 원래 문자열로 유지한다.
X_categorical = data_model[list(MODEL_FEATURE_NAMES)].copy()
y = data_model[TARGET_NAME].map({"Lost": 0, "Won": 1})
assert y.notna().all(), "Status에는 Lost와 Won만 있어야 합니다."

# Train/Test를 나누기 전에 원본 448행 전체에서 행별 고유 마스킹 10세트를 만든다.
X_all_masked_sets = make_unique_masked_sets(
    X_categorical,
    set_prefix="all",
    random_state=MASKING_RANDOM_STATE,
)

# 13개 입력이 완전히 같으면 같은 그룹으로 묶는다. Status는 그룹 생성에 사용하지 않는다.
input_group_ids = pd.Series(
    pd.util.hash_pandas_object(X_categorical, index=False).to_numpy(),
    index=X_categorical.index,
    name="input_group_id",
)

# 마스킹을 마친 뒤 10개의 층화 그룹 Fold 중 3개를 Test로 사용해 7:3을 만든다.
splitter = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=1)
group_folds = list(splitter.split(X_categorical, y, groups=input_group_ids))
test_positions = np.concatenate(
    [validation_positions for _, validation_positions in group_folds[:3]]
)
train_positions = np.setdiff1d(np.arange(len(X_categorical)), test_positions)

X_train_categorical = X_categorical.iloc[train_positions].copy()
X_test_categorical = X_categorical.iloc[test_positions].copy()
y_train_base = y.iloc[train_positions].copy()
y_test = y.iloc[test_positions].copy()
train_input_group_ids = input_group_ids.iloc[train_positions].copy()
test_input_group_ids = input_group_ids.iloc[test_positions].copy()

split_summary = pd.DataFrame(
    [
        {
            "데이터": "Train 원본",
            "행_수": len(X_train_categorical),
            "동일_입력_그룹_수": int(train_input_group_ids.nunique()),
            "Lost_수": int(y_train_base.eq(0).sum()),
            "Won_수": int(y_train_base.eq(1).sum()),
            "Won_비율(%)": float(y_train_base.mean() * 100),
        },
        {
            "데이터": "Test 원본",
            "행_수": len(X_test_categorical),
            "동일_입력_그룹_수": int(test_input_group_ids.nunique()),
            "Lost_수": int(y_test.eq(0).sum()),
            "Won_수": int(y_test.eq(1).sum()),
            "Won_비율(%)": float(y_test.mean() * 100),
        },
    ]
)
display(split_summary.round(3))

,데이터,행_수,동일_입력_그룹_수,Lost_수,Won_수,Won_비율(%)
0,Train 원본,313,138,154,159,50.799
1,Test 원본,135,60,67,68,50.370


### 해석

- 정답은 `Lost` 221행을 0, `Won` 227행을 1로 변환했다.
- 원본 448행 각각에서 Unknown 컬럼 조합이 겹치지 않는 10개 변형을 먼저 만들어 총 4,480개 마스킹 변형을 준비했다.
- 13개 입력 조합은 198개 그룹이며, 같은 입력 조합의 반복 행은 하나의 그룹에 유지한 채 Train 313행과 Test 135행으로 분리했다.
- Train은 `Lost` 154행, `Won` 159행이고 Test는 `Lost` 67행, `Won` 68행이다.
- Won 비율은 Train 50.80%, Test 50.37%로 거의 같다. Train과 Test 사이에는 동일한 13개 입력 조합이 없다.
- 중복 행을 제거한 것이 아니라 동일 입력 조합 138개 그룹은 Train에, 60개 그룹은 Test에 통째로 배정했다.

## 8. 마스킹 데이터의 원본 범주형 입력과 원핫 입력 준비

전체 448행에서 먼저 만든 마스킹 10세트를 동일한 그룹 분할 위치로 나눈다. 모델 단계에서는 원본 13개 범주형 입력을 공통으로 사용한다. 원핫 인코딩이 필요한 모델은 각 모델 파이프라인 안에서만 변환하고, 범주형을 직접 처리하는 모델은 원본 범주를 그대로 받는다.

In [16]:
# 전체 448행에서 만든 각 마스킹 세트를 같은 Train/Test 위치로 나눈다.
X_train_masked_sets = {
    f"train_mask_{set_number:02d}": masked_data.iloc[train_positions].copy()
    for set_number, masked_data in enumerate(X_all_masked_sets.values(), start=1)
}
X_test_masked_sets = {
    f"test_mask_{set_number:02d}": masked_data.iloc[test_positions].copy()
    for set_number, masked_data in enumerate(X_all_masked_sets.values(), start=1)
}

# 범주형 직접 학습 모델과 모델 내부 인코더가 공통으로 사용할 13개 컬럼 입력이다.
# pd.Categorical의 고정 범주 순서를 유지해 Train/Test와 모든 Fold의 범주 코드가 달라지지 않게 한다.
X_train_raw = pd.concat(
    X_train_masked_sets,
    names=["mask_set", "original_row_id"],
)
X_test_raw_sets = {
    set_name: masked_data.copy() for set_name, masked_data in X_test_masked_sets.items()
}

# 전처리 결과 확인과 원핫 모델의 입력 구조 검증을 위해 인코딩 결과도 함께 만든다.
X_train_sets = {
    set_name: encode_categories(masked_data)
    for set_name, masked_data in X_train_masked_sets.items()
}
X_test_sets = {
    set_name: encode_categories(masked_data) for set_name, masked_data in X_test_masked_sets.items()
}
model_input_columns = next(iter(X_train_sets.values())).columns.tolist()
X_train = pd.concat(
    X_train_sets,
    names=["mask_set", "original_row_id"],
)

# 정답과 그룹 ID는 원본 범주형 입력 및 원핫 입력과 같은 MultiIndex 순서로 반복한다.
y_train = pd.concat(
    {set_name: y_train_base for set_name in X_train_masked_sets},
    names=["mask_set", "original_row_id"],
)
train_original_row_ids = X_train_raw.index.get_level_values("original_row_id")
train_group_ids = input_group_ids.loc[train_original_row_ids].to_numpy()

masking_set_rows = []
for data_name, generation_seed, masked_sets, encoded_sets in (
    ("Train", MASKING_RANDOM_STATE, X_train_masked_sets, X_train_sets),
    ("Test", MASKING_RANDOM_STATE, X_test_masked_sets, X_test_sets),
):
    unique_mask_counts = unique_mask_counts_per_row(masked_sets)
    for set_number, (set_name, masked_data) in enumerate(masked_sets.items(), start=1):
        unknown_mask = masked_data.eq("Unknown")
        masking_set_rows.append(
            {
                "데이터": data_name,
                "세트": set_name,
                "세트_번호": set_number,
                "생성_시드": generation_seed,
                "행_수": len(masked_data),
                "행당_Unknown_최솟값": int(unknown_mask.sum(axis=1).min()),
                "행당_Unknown_최댓값": int(unknown_mask.sum(axis=1).max()),
                "Unknown_비율(%)": float(unknown_mask.to_numpy(dtype=bool).mean() * 100),
                "행별_고유_조합_최솟값": int(unique_mask_counts.min()),
                "행별_고유_조합_최댓값": int(unique_mask_counts.max()),
                "원본_범주형_컬럼_수": masked_data.shape[1],
                "원핫_컬럼_수": encoded_sets[set_name].shape[1],
            }
        )

masking_sets_summary = pd.DataFrame(masking_set_rows)
display(masking_sets_summary.round(3))
print(f"Train 원본 범주형 입력: {X_train_raw.shape[0]}행 × {X_train_raw.shape[1]}컬럼")
print(f"Train 원핫 확인 입력: {X_train.shape[0]}행 × {X_train.shape[1]}컬럼")
print(f"Test 마스킹 세트: {len(X_test_raw_sets)}개, 세트당 {len(y_test)}행")
display(pd.DataFrame({"원핫 입력 컬럼": model_input_columns}))

,데이터,세트,세트_번호,생성_시드,행_수,행당_Unknown_최솟값,행당_Unknown_최댓값,Unknown_비율(%),행별_고유_조합_최솟값,행별_고유_조합_최댓값,원본_범주형_컬럼_수,원핫_컬럼_수
0,Train,train_mask_01,1,1,313,4,4,30.769,10,10,13,39
1,Train,train_mask_02,2,1,313,4,4,30.769,10,10,13,39
2,Train,train_mask_03,3,1,313,4,4,30.769,10,10,13,39
3,Train,train_mask_04,4,1,313,4,4,30.769,10,10,13,39
4,Train,train_mask_05,5,1,313,4,4,30.769,10,10,13,39
5,Train,train_mask_06,6,1,313,4,4,30.769,10,10,13,39
6,Train,train_mask_07,7,1,313,4,4,30.769,10,10,13,39
7,Train,train_mask_08,8,1,313,4,4,30.769,10,10,13,39
8,Train,train_mask_09,9,1,313,4,4,30.769,10,10,13,39
9,Train,train_mask_10,10,1,313,4,4,30.769,10,10,13,39


Train 원본 범주형 입력: 3130행 × 13컬럼
Train 원핫 확인 입력: 3130행 × 39컬럼
Test 마스킹 세트: 10개, 세트당 135행


,원핫 입력 컬럼
0,Authority_Low
1,Authority_Mid
2,Authority_Unknown
3,Competitors_Unknown
4,Competitors_Yes
5,Purch_dept_Unknown
6,Purch_dept_Yes
7,Budgt_alloc_Unknown
8,Budgt_alloc_Yes
9,Forml_tend_Unknown


### 해석

- Train 313행의 마스킹 변형을 합쳐 `X_train_raw` 3,130행 × 13개 범주형 컬럼을 만들었다. Test는 `X_test_raw_sets` 10개로 분리해 유지한다.
- LogisticRegression, MultinomialNB, ExtraTrees는 모델 파이프라인 안에서 13개 범주를 39개 원핫 컬럼으로 바꾼다.
- CatBoost와 TabICL은 `Unknown`을 포함한 13개 범주형 컬럼을 직접 받는다. 두 모델에 사전 원핫 인코딩을 중복 적용하지 않는다.
- `X_train`과 `X_test_sets`는 39개 원핫 구조가 예상과 같은지 확인하기 위해 남겨 둔다. 두 표현의 행 순서·정답·그룹 ID는 동일하다.

In [17]:
# 전처리 결과가 예상한 데이터 구조와 일치하는지 마지막에 확인한다.
assert len(data_raw) == 448
assert int(full_duplicate_mask.sum()) == 83
assert int(model_duplicate_mask.sum()) == 226
assert len(data_model) == 448
assert int(conflicting_input_groups) == 24
assert not set(EXCLUDED_FEATURE_NAMES) & set(MODEL_FEATURE_NAMES)
assert not missing_masks(data_model)["effective"][TARGET_NAME].any()
assert set(data_model[TARGET_NAME].unique()) == {"Lost", "Won"}
assert set(y.unique()) == {0, 1}
assert len(X_train_categorical) == 313 and len(X_test_categorical) == 135
assert X_train_categorical.index.intersection(X_test_categorical.index).empty
assert set(train_input_group_ids).isdisjoint(set(test_input_group_ids))
assert input_group_ids.nunique() == len(pd.MultiIndex.from_frame(X_categorical).unique()) == 198
assert train_input_group_ids.nunique() == 138
assert test_input_group_ids.nunique() == 60
assert len(X_all_masked_sets) == MASKING_SET_COUNT
assert all(masked.shape == (448, 13) for masked in X_all_masked_sets.values())
assert unique_mask_counts_per_row(X_all_masked_sets).eq(MASKING_SET_COUNT).all()
assert len(X_train_masked_sets) == MASKING_SET_COUNT
assert len(X_test_masked_sets) == MASKING_SET_COUNT
for all_data, train_data, test_data in zip(
    X_all_masked_sets.values(),
    X_train_masked_sets.values(),
    X_test_masked_sets.values(),
    strict=True,
):
    assert train_data.equals(all_data.iloc[train_positions])
    assert test_data.equals(all_data.iloc[test_positions])
assert all(
    masked.eq("Unknown").sum(axis=1).eq(UNKNOWN_COLUMNS_PER_ROW).all()
    for masked in (*X_train_masked_sets.values(), *X_test_masked_sets.values())
)
train_mask_signatures = {
    masked.eq("Unknown").to_numpy(dtype=bool).tobytes() for masked in X_train_masked_sets.values()
}
test_mask_signatures = {
    masked.eq("Unknown").to_numpy(dtype=bool).tobytes() for masked in X_test_masked_sets.values()
}
assert len(train_mask_signatures) == MASKING_SET_COUNT
assert len(test_mask_signatures) == MASKING_SET_COUNT
assert unique_mask_counts_per_row(X_train_masked_sets).eq(MASKING_SET_COUNT).all()
assert unique_mask_counts_per_row(X_test_masked_sets).eq(MASKING_SET_COUNT).all()
expected_unknown_columns = {f"{column}_Unknown" for column in MODEL_FEATURE_NAMES}
assert expected_unknown_columns <= set(model_input_columns)
assert X_train_raw.shape == (3130, 13)
assert X_train.shape == (3130, 39)
assert X_train_raw.index.equals(X_train.index)
assert X_train_raw.index.equals(y_train.index)
assert all(str(dtype) == "category" for dtype in X_train_raw.dtypes)
assert not X_train.select_dtypes(include=["object", "string"]).columns.any()
assert all(
    raw_data.shape == (135, 13)
    and raw_data.columns.tolist() == list(MODEL_FEATURE_NAMES)
    and raw_data.index.equals(y_test.index)
    and all(str(dtype) == "category" for dtype in raw_data.dtypes)
    for raw_data in X_test_raw_sets.values()
)
assert all(
    encoded_data.shape == (135, 39)
    and encoded_data.columns.tolist() == model_input_columns
    and encoded_data.index.equals(y_test.index)
    for encoded_data in X_test_sets.values()
)
train_group_counts = pd.Series(train_group_ids).value_counts()
assert len(train_group_counts) == 138 and train_group_counts.mod(10).eq(0).all()
print("데이터 전처리 검증을 통과했습니다.")

데이터 전처리 검증을 통과했습니다.


### 해석

- 원본 448행 전체에서 행별 고유 마스킹 10세트를 먼저 만든 뒤 동일 입력 그룹을 유지해 Train/Test로 나눴다.
- 학습용 결과는 `X_train_raw`, `y_train`, `train_group_ids`이며 평가용 결과는 `X_test_raw_sets` 10개와 공통 `y_test`다.
- 원본 범주형 13개와 원핫 39개의 행·인덱스가 정확히 대응하므로 모델마다 필요한 표현을 사용해도 비교 대상은 달라지지 않는다.
- 원본 CSV는 수정하지 않았다.